In [7]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df = pd.read_csv(r"C:\Users\ganes\Downloads\Resume.zip")   
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\r', ' ', text)
    text = re.sub(f"[{string.punctuation}]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)
skills_list = [
    "python", "java", "c++", "machine learning", "deep learning",
    "sql", "excel", "data analysis", "nlp", "tensorflow",
    "pandas", "numpy", "power bi", "tableau", "aws"
]
def extract_skills(text):
    found_skills = []
    for skill in skills_list:
        if skill in text:
            found_skills.append(skill)
    return list(set(found_skills))

df['skills'] = df['cleaned_resume'].apply(extract_skills)
job_description = """
Looking for a Data Scientist with skills in Python, Machine Learning,
SQL, Data Analysis, and Deep Learning
"""
cleaned_job = clean_text(job_description)
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['cleaned_resume'])
job_vector = tfidf.transform([cleaned_job])
similarity_scores = cosine_similarity(job_vector, tfidf_matrix)
df['similarity_score'] = similarity_scores.flatten()
job_skills = extract_skills(cleaned_job)
def skill_gap(candidate_skills):
    return list(set(job_skills) - set(candidate_skills))
df['missing_skills'] = df['skills'].apply(skill_gap)
df_sorted = df.sort_values(by='similarity_score', ascending=False)
for i, row in df_sorted.head(top_n).iterrows():
    print("="*50)
    print(f"Candidate ID: {row['ID']}")
    print(f"Match Score: {round(row['similarity_score']*100, 2)}%")
    print(f"Skills: {row['skills']}")
    print(f"Missing Skills: {row['missing_skills']}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ganes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Candidate ID: 12011623
Match Score: 21.81%
Skills: ['pandas', 'excel', 'tableau', 'python', 'data analysis', 'sql', 'machine learning']
Missing Skills: ['deep learning']
Candidate ID: 21156767
Match Score: 20.51%
Skills: ['java', 'python', 'data analysis', 'sql', 'machine learning']
Missing Skills: ['deep learning']
Candidate ID: 34953092
Match Score: 19.38%
Skills: ['machine learning', 'python', 'sql']
Missing Skills: ['deep learning', 'data analysis']
Candidate ID: 12777487
Match Score: 17.03%
Skills: []
Missing Skills: ['python', 'sql', 'data analysis', 'machine learning', 'deep learning']
Candidate ID: 18448085
Match Score: 16.19%
Skills: ['pandas', 'excel', 'tableau', 'numpy', 'python', 'data analysis', 'sql']
Missing Skills: ['machine learning', 'deep learning']


In [3]:
print(df.columns)

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')
